# Capture unification demo

This notebook demonstrates the unified `save=` / `intervene=` capture surface on tiny CPU models.

## In-flight vs postprocess facts

| Backend | Knowable during capture | Only known after postprocess |
| --- | --- | --- |
| Torch eager | Function name, raw label, raw index, shape, dtype, device, requires-grad, module stack, raw parents, recent events, timing, scalar-bool value | Final label, ordinal, loop membership, recurrence grouping, module-path-qualified labels |
| MLX lazy | Shape, dtype, and device at call time; value-dependent facts require forced evaluation | Final label, ordinal, loop membership, recurrence grouping, module-path-qualified labels |

The practical consequence is that `save=` predicates work during the forward pass, while final-label selection through `layers_to_save` remains a two-pass path.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import sys
import tempfile
import warnings

import torch
from torch import nn

repo_root = Path.cwd().resolve()
if not (repo_root / "torchlens").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import torchlens as tl  # noqa: E402

torch.manual_seed(0)
warnings.filterwarnings(
    "ignore",
    message="TorchLens intervention-ready output traversal does not support UntypedStorage.*",
    category=UserWarning,
)
OUT = Path(tempfile.mkdtemp(prefix="torchlens_capture_unification_"))
print(f"torchlens {tl.__version__}; tmp={OUT.name}")

In [ ]:
class ConvRelu(nn.Module):
    """Tiny convolutional model with conv and relu ops."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run conv, relu, and add."""

        return torch.relu(self.conv(x)) + 1


class LinearRelu(nn.Module):
    """Tiny named-linear model for interventions."""

    def __init__(self) -> None:
        """Initialize deterministic weights."""

        super().__init__()
        self.attn = nn.Linear(4, 4)
        with torch.no_grad():
            self.attn.weight.copy_(torch.eye(4))
            self.attn.bias.fill_(1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run a linear site followed by relu."""

        return torch.relu(self.attn(x))


class Layer4Predecessor(nn.Module):
    """Model whose relu has a predecessor inside layer4."""

    def __init__(self) -> None:
        """Initialize deterministic weights."""

        super().__init__()
        self.layer4 = nn.Linear(4, 4)
        with torch.no_grad():
            self.layer4.weight.copy_(torch.eye(4))
            self.layer4.bias.fill_(2.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run layer4 then relu."""

        hidden = self.layer4(x)
        return torch.relu(hidden)


class RecurrentAttn(nn.Module):
    """Repeated module model for layer-label pass behavior."""

    def __init__(self, passes: int = 3) -> None:
        """Initialize recurrent module."""

        super().__init__()
        self.attn = nn.Linear(4, 4)
        self.passes = passes

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the same module repeatedly."""

        for _ in range(self.passes):
            x = torch.relu(self.attn(x))
        return x


x_img = torch.randn(1, 1, 4, 4)
x_vec = torch.ones(1, 4)

## 1. Save by function predicate

In [ ]:
relu_trace = tl.trace(ConvRelu(), x_img, save=tl.func("relu"), random_seed=1)
saved_relu = [op.label for op in relu_trace.layer_list if op.has_saved_activation]
print(saved_relu)
assert any("relu" in label for label in saved_relu)

## 2. Windowed retroactive save

In [ ]:
windowed = tl.trace(
    ConvRelu(),
    x_img,
    save=tl.func("conv2d") & tl.followed_by(tl.func("relu")),
    lookback=4,
    lookback_payload_policy="detached_raw",
    random_seed=2,
)
saved_windowed = [op.label for op in windowed.layer_list if op.has_saved_activation]
print(saved_windowed)
assert any("conv2d" in label for label in saved_windowed)

## 3. Compose intervention and save

In [ ]:
patched = tl.trace(
    LinearRelu(),
    x_vec,
    intervene=tl.when(tl.func("linear"), tl.scale(0.5)),
    save=tl.func("linear") | tl.func("relu"),
    random_seed=3,
)
linear_op = next(op for op in patched.layer_list if op.layer_type == "linear")
relu_op = next(op for op in patched.layer_list if op.layer_type == "relu")
print(linear_op.label, relu_op.label, linear_op.intervention_replaced)
assert linear_op.intervention_replaced
assert torch.equal(relu_op.out, torch.relu(linear_op.out))

## 4. Stream selected payloads to disk

In [ ]:
bundle_path = OUT / "streamed.tlspec"
streamed = tl.trace(
    LinearRelu(),
    x_vec,
    save=tl.in_module("attn"),
    storage=tl.to_disk(bundle_path),
    random_seed=4,
)
streamed_saved = [op for op in streamed.layer_list if op.has_saved_activation]
print(bundle_path.exists(), [op.label for op in streamed_saved])
assert bundle_path.exists()
assert streamed_saved

## 5. Sparse record and materialize to Trace

In [ ]:
recording = tl.record(ConvRelu(), x_img, save=tl.func("conv2d"), random_seed=5)
print(len(recording.records), isinstance(recording, tl.CapturedRun))
recorded_trace = recording.to_trace()
print(len(recorded_trace.layer_list), isinstance(recorded_trace, tl.Trace))
assert len(recording.records) == 1
assert isinstance(recorded_trace, tl.Trace)

## 6. Conditional intervention with predecessor and module predicates

In [ ]:
conditional = tl.trace(
    Layer4Predecessor(),
    x_vec,
    intervene=tl.when(
        tl.func("relu") & tl.preceded_by(tl.in_module("layer4")),
        tl.zero_ablate(),
    ),
    save=tl.func("linear") | tl.func("relu"),
    lookback=4,
    random_seed=6,
)
conditional_relu = next(op for op in conditional.layer_list if op.layer_type == "relu")
print(conditional_relu.label, conditional_relu.intervention_replaced)
assert conditional_relu.intervention_replaced
assert torch.equal(conditional_relu.out, torch.zeros_like(conditional_relu.out))

## 7. Layer labels save all passes, pass labels save one pass

In [ ]:
all_passes = tl.trace(RecurrentAttn(passes=3), x_vec, layers_to_save=["attn"], random_seed=7)
one_pass = tl.trace(RecurrentAttn(passes=3), x_vec, layers_to_save=["attn:2"], random_seed=7)
all_linear = next(layer for layer in all_passes.layers if layer.layer_type == "linear")
one_linear = next(layer for layer in one_pass.layers if layer.layer_type == "linear")
print([op.has_saved_activation for op in all_linear.ops._list])
print([op.has_saved_activation for op in one_linear.ops._list])
assert [op.has_saved_activation for op in all_linear.ops._list] == [True, True, True]
assert [op.has_saved_activation for op in one_linear.ops._list] == [False, True, False]

In [ ]:
shutil.rmtree(OUT)
print("done")